# Exploratory Data Analysis

This notebook uses the cleaned data from the folder data/processed and visualizes it (e.g. via Pandas Profiling).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport

from c08_farming_exit import config, visuals

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [51]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Visualizations

### 2.1 Off-Farm and On-Farm Distributions

In [ ]:
#EMPLOYMENT CATEGORIES
conditions = [
    (df["farm_empl_last_12_months"] == "Yes") & (df["empl_type"].isnull()),
    (df["farm_empl_last_12_months"] == "Yes") & (df["empl_type"].notnull()),
    (df["farm_empl_last_12_months"] == "No")  & (df["empl_type"].notnull()),
]
choices = ["only farm", "hybrid", "fully off-farm"]

df["empl_category"] = np.select(conditions, choices, default="noise")

visuals.plot_stacked_bar(df, "empl_category",
                 title="Employment Categories by Country",
                 legend_title="Employment Category",
                 filename="employment_categories_by_country.png")

In [ ]:
#EMPLOYMENT TYPES
loop = ["fully off-farm", "hybrid"]

for i in loop:
    subset = df[df["empl_category"] == i][["country", "personal_id", "empl_type"]]

    visuals.plot_stacked_bar(subset, "empl_type",
                    title=f"Only {i}: Employment Types",
                    legend_title="Employment Type",
                    filename=f"{i.replace(' ', '_')}_employment_types_by_country.png")

In [ ]:
#OCCUPATION SECTORS
loop_1 = ["fully off-farm", "hybrid"]
loop_2 = ["Self-employed/own business", "Employee"]

current_sector_map = {
    #Primary Sector: creating raw materials
    "Small-scale farm": "Agriculture",
    "Large-scale farm": "Agriculture",

    #Secondary Sector: turning raw materials into goods -- NONE

    #Tertiary Sector: Services (no production)
    "Retail trade": "Private Service",
    "Beauty industry (hair, skin etc)": "Private Service",
    "Food industry": "Private Service",
    "Security services": "Private Service",
    "Transport": "Private Service",
    "Hospitality (e.g., Accommodation & Lodging )": "Private Service",
    "Domestic/household helper": "Private Service",
    "Travel and Tourism": "Private Service",
    "Recreation & Events": "Private Service",
    "Education (teacher etc)": "Public Service",
    "Health (nurse, doctor etc)": "Public Service",
    "Other public sector": "Public Service",

    #Other/Unclear
    "Others, specify": "Other/Unclear",
    "I don't know": "Other/Unclear",
}


for i in loop_1:
    for j in loop_2:
        subset = df[(df["empl_category"] == i) & (df["empl_type"] == j)][["country", "personal_id", "sector_off_farm_empl_last_12_months"]]
        subset["sector_off_farm_empl_last_12_months"] = subset["sector_off_farm_empl_last_12_months"].map(current_sector_map)

        visuals.plot_stacked_bar(subset, "sector_off_farm_empl_last_12_months",
                        title=f"Only {i} and {j}: Employment Sectors",
                        legend_title="Employment Sector",
                        filename=f"{i.replace(' ', '_')}_{j.replace(' ', '_').replace('/', '-')}_employment_sectors_by_country.png")


### 2.2 Target Variables

In [ ]:
# TARGET 1: ASPIRATION TO STAY OR LEAVE FARMING BY COUNTRY
loop = ["only farm", "hybrid", "fully off-farm"]

for i in loop:
    subset = df[df["empl_category"] == i][["country", "personal_id", "aspiration_continue_farming"]]

    mapping = {
        "Continue to farming":             "continue farming",
        "Both":                            "both",
        "Other non-agricultural business": "exiting farming",
        "Not engaged in farming":          "exiting farming",
    }

    subset["aspiration_continue_farming"] = subset["aspiration_continue_farming"].map(mapping)

    visuals.plot_stacked_bar(subset, "aspiration_continue_farming",
                    title=f"{i}: Aspiration to Stay or Leave Farming (Target 1)",
                    legend_title="Aspiration",
                    filename=f"{i.replace(' ', '_')}_stay_or_leave_farming_by_country.png")
    

In [ ]:
# TARGET 2:ASPIRED OCCUPATION GROUPS 5 YEARS AHEAD BY COUNTRY
loop = ["only farm", "hybrid", "fully off-farm"]

for i in loop:

    subset = df[df["empl_category"] == i] \
            [["country", "personal_id", "aspired_occupation_5_years_ahead"]]

    aspired_occupation_map = {
        #Primary Sector: creating raw materials
        "Crop farming/cultivation": "Agriculture",
        "Both crop and livestock/fish farming": "Agriculture",
        "Livestock keeping/raising": "Agriculture",
        "Fishing": "Agriculture",
        "Agricultural wage labour (hired agricultural labour)": "Agriculture",
        "Self-employed agribusiness/agrienterprise": "Agriculture",

        #Secondary Sector: turning raw materials into goods
        "Factory worker/manufacturing jobs": "Manufacturing",
        "Construction labour": "Manufacturing",
        
        #Tertiary Sector: Services (no production)
        "Self-employed non-agribusiness (wholesale/retail trade, etc.)": "Private Service",
        "Other service sector jobs (worker in hotels, restaurants, shops, security guards, etc.)": "Private Service",
        "Domestic/household helper": "Private Service",
        "Driver/transport": "Private Service",
        "Government officers/employees/ civil servant": "Public Service",
        "Teacher/education": "Public Service",

        #Other/Unclear
        "Charcoal burning, production, and selling": "Charcoal",
        "None/No employment": "Other/Unclear",
        "Retired/Pensioner": "Other/Unclear",
    }

    subset["aspired_occupation_5_years_ahead"] = subset["aspired_occupation_5_years_ahead"].map(aspired_occupation_map)


    visuals.plot_stacked_bar(subset, "aspired_occupation_5_years_ahead",
                    title=f"{i}: Occupational Aspirations by Country (Target 2)",
                    legend_title="Aspiration",
                    filename=f"{i.replace(' ', '_')}_occupational_aspirations_by_country.png")

In [ ]:
# TARGET 3: ASPIRED SELF-EMPLYMENT 5 YEARS AHEAD BY COUNTRY
loop = ["only farm", "hybrid", "fully off-farm"]

for i in loop:

    subset = df[df["empl_category"] == i] \
            [["country", "personal_id", "aspired_occupation_5_years_ahead"]]

    aspired_self_employment_map = {
        #Primary Sector: creating raw materials
        "Crop farming/cultivation": "No Self-Employment",
        "Both crop and livestock/fish farming": "No Self-Employment",
        "Livestock keeping/raising": "No Self-Employment",
        "Fishing": "No Self-Employment",
        "Agricultural wage labour (hired agricultural labour)": "No Self-Employment",
        "Self-employed agribusiness/agrienterprise": "Self-Employment: Agriculture",

        #Secondary Sector: turning raw materials into goods
        "Factory worker/manufacturing jobs": "No Self-Employment",
        "Construction labour": "No Self-Employment",
        
        #Tertiary Sector: Services (no production)
        "Self-employed non-agribusiness (wholesale/retail trade, etc.)": "Self-Employment: Non-Agriculture",
        "Other service sector jobs (worker in hotels, restaurants, shops, security guards, etc.)": "No Self-Employment",
        "Domestic/household helper": "No Self-Employment",
        "Driver/transport": "No Self-Employment",
        "Government officers/employees/ civil servant": "No Self-Employment",
        "Teacher/education": "No Self-Employment",

        #Other/Unclear
        "Charcoal burning, production, and selling": "No Self-Employment",
        "None/No employment": "No Self-Employment",
        "Retired/Pensioner": "No Self-Employment",
    }

    subset["aspired_self_employment_5_years_ahead"] = subset["aspired_occupation_5_years_ahead"].map(aspired_self_employment_map)


    visuals.plot_stacked_bar(subset, "aspired_self_employment_5_years_ahead",
                    title=f"{i}: Self-Employement Aspirations by Country (Target 3)",
                    legend_title="Aspiration",
                    filename=f"{i.replace(' ', '_')}_self_employment_aspirations_by_country.png")

In [ ]:
# TARGET 4: CHILD ASPIRATION TO STAY OR LEAVE FARMING BY COUNTRY
loop = ["only farm", "hybrid", "fully off-farm"]

for i in loop:
    subset = df[df["empl_category"] == i][["country", "personal_id", "most_common_child_aspiration"]]

    mapping = {
        "Likely":               "Likely",
        "Maybe, maybe not":     "Maybe, maybe not",
        "Unlikely":             "Unlikely",
        "Very likely":          "Likely",
        "Very unlikely":        "Unlikely",
    }

    subset["most_common_child_aspiration"] = subset["most_common_child_aspiration"].map(mapping)
    
    visuals.plot_stacked_bar(subset, "most_common_child_aspiration",
                    title=f"{i}: Child Aspiration to Continue Farming (Target 4)",
                    legend_title="Aspiration",
                    filename=f"{i.replace(' ', '_')}_child_aspiration_stay_or_leave_farming_by_country.png")

## 3. Pandas Profiling

In [59]:
profile = ProfileReport(df, title="Farming Exit")
profile.to_file("../output/pandas_profiling.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]